### Output Parser
Output parser is a responsible for taking the output of a model and transforming it to a more suitable format for downstram tasks. Useful when you are using LLMs to generate structured data or to normalize output from chat models and LLMs.

when to use with_structured_output() and output_parser

with_structured_output() - when LLM is capable of giving output in structured way
output_parser -            when LLM is not capable of giving output in structured way

Type of Parser :- 1)String  2)CSV  3)JSON  4)OutputFixing  5)XML  6)RetryWithError  7)Pydantic  8)YAML  9)Structured 10)Manymore

### StringOutputParser
No formating is required , Best when you just want the LLMs text output as it is. , No structured or schema enforcement.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
from json import load
load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.6)
## First example: simple string output using traditional prompt template


template = PromptTemplate.from_template("Who is the president of the country {country}?")
chat = template.invoke({"country": "United States"})

result = llm.invoke(chat)
print(result.content)

content='The current president of the United States is **Joe Biden**.' additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019eb264-ea7f-74b3-939f-35331aa6c2b3-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 11, 'output_tokens': 58, 'total_tokens': 69, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 46}}


### Using Parser

In [ ]:
parser = StrOutputParser() # parser automatically call content attribute of the response object
template = PromptTemplate.from_template("Who is the President of countrry {country}")

chat = template.invoke({"country": "United States"})
result = llm.invoke(chat)
final_result = parser.invoke(result)
print(final_result)

The current President of the United States is **Joe Biden**.


### With chain
In LangChain, chains act as step-by-step workflows that complete tasks in sequence.

In [4]:
parser = StrOutputParser()
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.6)
template = PromptTemplate.from_template("Who is the President of countrry {country}")

chain = template | llm | parser

result = chain.invoke({"country": "India"})

print(result)

The current President of India is **Droupadi Murmu**.


### Json Output Parser

In [6]:
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import JsonOutputParser
json_parser = JsonOutputParser()

template = PromptTemplate.from_template("""
Extract the following product information from the text and retun it as a JSON:
- Name
-Price
- Category
-features (list)

text: {text}
format_instructions: {format_instructions}                                                                                                           
""",

partial_variables= {"format_instructions": json_parser.get_format_instructions() }
                                        
) #here the .get_format_instructions() method is taken from the JsonOutputParser class, which consist of the instructions

chain = template | llm | json_parser

text = """
The Samsung Galaxy S24 Ultra was launched at $1199. 
It belongs to the smartphone category. 
Main features include a 200MP camera, S-Pen support, Snapdragon 8 Gen 3 processor, and a 5000mAh battery.
"""
result = chain.invoke({"text": text})
print(result)

{'Name': 'Samsung Galaxy S24 Ultra', 'Price': '$1199', 'Category': 'smartphone', 'features': ['200MP camera', 'S-Pen support', 'Snapdragon 8 Gen 3 processor', '5000mAh battery']}


### Pydantic Output Parser

In [ ]:
from platform import release # to check the version of the platform
from langchain_core.output_parsers import PydanticOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel, Field
from typing import List, Optional

class Movie(BaseModel):
  title:str = Field(..., description="The title of the movie")
  release_year: int = Field(..., description="Year the movie was released")
  genres: List[str] = Field(..., description="List of genres the movie belongs to")
  rating: float = Field(..., description="Average rating of the movie on a scale of 1 to 10")
  box_office: Optional[float] = Field(None, description="Worldwide box office in millions USD, if known")


llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.6)

pydantic_parser = PydanticOutputParser(pydantic_object=Movie)

template = PromptTemplate.from_template("""Extract movie information from the folloing text: 
{text}
{format_instructions}""",
partial_variables={"format_instructions": pydantic_parser.get_format_instructions()})

chain = template | llm | pydantic_parser

text = """
The movie 'Inception', directed by Christopher Nolan, was released in 2010. 
It falls under the genres of science fiction, action, and thriller. 
It holds an average rating of 8.8 out of 10. 
The film grossed approximately $829.9 million at the global box office.
"""

result = chain.invoke({"text": text})
print(result)

d:\LangChainP\myenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


title='Inception' release_year=2010 genres=['science fiction', 'action', 'thriller'] rating=8.8 box_office=829.9


: 